# 🖼️ Convolutional Neural Networks (CNN) From Scratch

> Understanding CNNs by implementing convolution, pooling, and feature extraction using NumPy

**Topics covered:**
- Convolution operation
- Padding and stride
- Max pooling and average pooling
- Feature maps visualization
- Building a simple CNN classifier
- Using PyTorch CNN for comparison

## 📚 Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# For PyTorch comparison (optional)
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    print("✅ PyTorch available")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️ PyTorch not available - NumPy only mode")

np.random.seed(42)
print("✅ All imports successful!")

## 🔧 Part 1: Convolution Operation From Scratch

In [ ]:
class Conv2D:
    """
    2D Convolution layer implemented from scratch using NumPy
    
    Input: (batch_size, height, width, channels)
    Output: (batch_size, out_height, out_width, num_filters)
    """
    
    def __init__(self, num_filters, kernel_size, input_channels, stride=1, padding=0):
        """
        Initialize convolution layer
        
        Args:
            num_filters: Number of output filters/kernels
            kernel_size: Size of the square kernel (e.g., 3 for 3x3)
            input_channels: Number of input channels
            stride: Step size for sliding window
            padding: Zero padding around input
        """
        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.input_channels = input_channels
        self.stride = stride
        self.padding = padding
        
        # Initialize kernels with He initialization
        # Shape: (kernel_size, kernel_size, input_channels, num_filters)
        self.kernels = np.random.randn(
            kernel_size, kernel_size, input_channels, num_filters
        ) * np.sqrt(2.0 / (kernel_size * kernel_size * input_channels))
        
        # Bias for each filter
        self.biases = np.zeros((1, 1, 1, num_filters))
        
        # For backpropagation
        self.last_input = None
        self.last_output = None
    
    def _pad_input(self, x):
        """Apply zero padding to input"""
        if self.padding > 0:
            return np.pad(
                x,
                ((0, 0), (self.padding, self.padding), (self.padding, self.padding), (0, 0)),
                mode='constant'
            )
        return x
    
    def forward(self, x):
        """
        Forward pass of convolution
        
        Args:
            x: Input of shape (batch, height, width, channels)
        
        Returns:
            Convolved output
        """
        self.last_input = x
        
        batch_size, h, w, c = x.shape
        
        # Pad input
        x_padded = self._pad_input(x)
        
        # Calculate output dimensions
        out_h = (h + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_w = (w + 2 * self.padding - self.kernel_size) // self.stride + 1
        
        # Initialize output
        output = np.zeros((batch_size, out_h, out_w, self.num_filters))
        
        # Perform convolution
        for b in range(batch_size):
            for i in range(out_h):
                for j in range(out_w):
                    # Define the region of interest
                    h_start = i * self.stride
                    h_end = h_start + self.kernel_size
                    w_start = j * self.stride
                    w_end = w_start + self.kernel_size
                    
                    # Extract region
                    region = x_padded[b, h_start:h_end, w_start:w_end, :]
                    
                    # Element-wise multiplication and sum
                    for f in range(self.num_filters):
                        output[b, i, j, f] = np.sum(region * self.kernels[:, :, :, f]) + self.biases[0, 0, 0, f]
        
        self.last_output = output
        return output
    
    def backward(self, d_out, learning_rate):
        """
        Backward pass of convolution
        
        Args:
            d_out: Gradient from next layer
            learning_rate: Learning rate for updates
        """
        batch_size, h, w, c = self.last_input.shape
        
        # Initialize gradients
        d_kernels = np.zeros_like(self.kernels)
        d_biases = np.zeros_like(self.biases)
        d_input = np.zeros_like(self._pad_input(self.last_input))
        
        x_padded = self._pad_input(self.last_input)
        out_h, out_w = d_out.shape[1], d_out.shape[2]
        
        # Compute gradients
        for b in range(batch_size):
            for i in range(out_h):
                for j in range(out_w):
                    h_start = i * self.stride
                    h_end = h_start + self.kernel_size
                    w_start = j * self.stride
                    w_end = w_start + self.kernel_size
                    
                    region = x_padded[b, h_start:h_end, w_start:w_end, :]
                    
                    for f in range(self.num_filters):
                        d_kernels[:, :, :, f] += region * d_out[b, i, j, f]
                        d_biases[0, 0, 0, f] += d_out[b, i, j, f]
                        d_input[b, h_start:h_end, w_start:w_end, :] += 
                            self.kernels[:, :, :, f] * d_out[b, i, j, f]
        
        # Remove padding from d_input if needed
        if self.padding > 0:
            d_input = d_input[:, self.padding:-self.padding, self.padding:-self.padding, :]
        
        # Update parameters
        self.kernels -= learning_rate * d_kernels / batch_size
        self.biases -= learning_rate * d_biases / batch_size
        
        return d_input
    
    def get_feature_maps(self):
        """Return the kernels for visualization"""
        return self.kernels

# Test convolution
print("=== Testing Conv2D ===")
# Create a simple 8x8 image with 1 channel
test_image = np.random.randn(1, 8, 8, 1)

# Create conv layer: 2 filters, 3x3 kernel, 1 input channel
conv = Conv2D(num_filters=2, kernel_size=3, input_channels=1, stride=1, padding=1)

# Forward pass
output = conv.forward(test_image)
print(f"Input shape: {test_image.shape}")
print(f"Output shape: {output.shape}")
print(f"Kernels shape: {conv.kernels.shape}")
print("✅ Conv2D working!")

## 🏊 Part 2: Pooling Layers

In [ ]:
class MaxPool2D:
    """
    Max Pooling layer - reduces spatial dimensions
    Keeps only the maximum value in each pooling window
    """
    
    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride = stride
        self.last_input = None
        self.max_indices = None
    
    def forward(self, x):
        """
        Forward pass of max pooling
        
        Args:
            x: Input of shape (batch, height, width, channels)
        """
        self.last_input = x
        batch_size, h, w, c = x.shape
        
        # Calculate output dimensions
        out_h = (h - self.pool_size) // self.stride + 1
        out_w = (w - self.pool_size) // self.stride + 1
        
        output = np.zeros((batch_size, out_h, out_w, c))
        self.max_indices = np.zeros((batch_size, out_h, out_w, c, 2), dtype=int)
        
        for b in range(batch_size):
            for i in range(out_h):
                for j in range(out_w):
                    for ch in range(c):
                        # Define pooling region
                        h_start = i * self.stride
                        h_end = h_start + self.pool_size
                        w_start = j * self.stride
                        w_end = w_start + self.pool_size
                        
                        region = x[b, h_start:h_end, w_start:w_end, ch]
                        
                        # Max value and its position
                        max_val = np.max(region)
                        max_pos = np.unravel_index(np.argmax(region), region.shape)
                        
                        output[b, i, j, ch] = max_val
                        self.max_indices[b, i, j, ch] = [h_start + max_pos[0], w_start + max_pos[1]]
        
        return output
    
    def backward(self, d_out):
        """
        Backward pass - gradient flows only through max positions
        """
        batch_size, h, w, c = self.last_input.shape
        d_input = np.zeros_like(self.last_input)
        
        out_h, out_w = d_out.shape[1], d_out.shape[2]
        
        for b in range(batch_size):
            for i in range(out_h):
                for j in range(out_w):
                    for ch in range(c):
                        h_idx, w_idx = self.max_indices[b, i, j, ch]
                        d_input[b, h_idx, w_idx, ch] += d_out[b, i, j, ch]
        
        return d_input


class AveragePool2D:
    """Average Pooling - takes average of each pooling window"""
    
    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride = stride
        self.last_input = None
    
    def forward(self, x):
        self.last_input = x
        batch_size, h, w, c = x.shape
        
        out_h = (h - self.pool_size) // self.stride + 1
        out_w = (w - self.pool_size) // self.stride + 1
        
        output = np.zeros((batch_size, out_h, out_w, c))
        
        for b in range(batch_size):
            for i in range(out_h):
                for j in range(out_w):
                    for ch in range(c):
                        h_start = i * self.stride
                        h_end = h_start + self.pool_size
                        w_start = j * self.stride
                        w_end = w_start + self.pool_size
                        
                        region = x[b, h_start:h_end, w_start:w_end, ch]
                        output[b, i, j, ch] = np.mean(region)
        
        return output
    
    def backward(self, d_out):
        """Distribute gradient evenly to all positions in pooling window"""
        batch_size, h, w, c = self.last_input.shape
        d_input = np.zeros_like(self.last_input)
        
        out_h, out_w = d_out.shape[1], d_out.shape[2]
        
        for b in range(batch_size):
            for i in range(out_h):
                for j in range(out_w):
                    for ch in range(c):
                        h_start = i * self.stride
                        h_end = h_start + self.pool_size
                        w_start = j * self.stride
                        w_end = w_start + self.pool_size
                        
                        d_input[b, h_start:h_end, w_start:w_end, ch] += 
                            d_out[b, i, j, ch] / (self.pool_size ** 2)
        
        return d_input

# Test pooling
print("=== Testing Pooling ===")
test_input = np.random.randn(1, 8, 8, 2)

max_pool = MaxPool2D(pool_size=2, stride=2)
avg_pool = AveragePool2D(pool_size=2, stride=2)

max_output = max_pool.forward(test_input)
avg_output = avg_pool.forward(test_input)

print(f"Input shape: {test_input.shape}")
print(f"MaxPool output: {max_output.shape}")
print(f"AvgPool output: {avg_output.shape}")
print("✅ Pooling layers working!")

## 🎨 Part 3: Visualizing Convolution & Feature Maps

In [ ]:
# Create a simple image with patterns
def create_test_image(size=16):
    """Create a test image with edges and patterns"""
    image = np.zeros((size, size))
    
    # Add a square
    image[4:12, 4:12] = 1.0
    
    # Add a diagonal line
    for i in range(size):
        if i < size:
            image[i, i] = 0.5
    
    # Add noise
    image += np.random.randn(size, size) * 0.1
    
    return image

# Create test image
test_img = create_test_image(16)
test_img_batch = test_img.reshape(1, 16, 16, 1)

# Define edge detection kernels
edge_kernels = np.array([
    # Horizontal edge
    [[[1, 1, 1], [0, 0, 0], [-1, -1, -1]]],
    # Vertical edge
    [[[1, 0, -1], [1, 0, -1], [1, 0, -1]]],
    # Diagonal edge
    [[[0, 1, 1], [-1, 0, 1], [-1, -1, 0]]],
    # Sobel horizontal
    [[[1, 2, 1], [0, 0, 0], [-1, -2, -1]]]
]).transpose(2, 3, 0, 1)  # Shape: (3, 3, 1, 4)

# Create conv layer with these kernels
conv_viz = Conv2D(num_filters=4, kernel_size=3, input_channels=1, stride=1, padding=1)
conv_viz.kernels = edge_kernels

# Forward pass
feature_maps = conv_viz.forward(test_img_batch)

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Original image
axes[0, 0].imshow(test_img, cmap='gray')
axes[0, 0].set_title('Original Image', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Feature maps
titles = ['Horizontal Edge', 'Vertical Edge', 'Diagonal Edge', 'Sobel Horizontal']
for i in range(4):
    row = (i + 1) // 3
    col = (i + 1) % 3
    axes[row, col].imshow(feature_maps[0, :, :, i], cmap='viridis')
    axes[row, col].set_title(titles[i], fontsize=12, fontweight='bold')
    axes[row, col].axis('off')

# Remove empty subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.suptitle('Convolution Feature Maps', fontsize=16, fontweight='bold', y=1.02)
plt.show()

print("✅ Feature maps visualized!")

## 🏗️ Part 4: Complete CNN Classifier

In [ ]:
class Flatten:
    """Flatten layer: (batch, h, w, c) -> (batch, h*w*c)"""
    
    def __init__(self):
        self.input_shape = None
    
    def forward(self, x):
        self.input_shape = x.shape
        return x.reshape(x.shape[0], -1)
    
    def backward(self, d_out):
        return d_out.reshape(self.input_shape)


class Dense:
    """Fully connected (dense) layer"""
    
    def __init__(self, input_size, output_size):
        self.weights = np.random.randn(input_size, output_size) * np.sqrt(2.0 / input_size)
        self.biases = np.zeros((1, output_size))
        self.last_input = None
    
    def forward(self, x):
        self.last_input = x
        return np.dot(x, self.weights) + self.biases
    
    def backward(self, d_out, learning_rate):
        d_weights = np.dot(self.last_input.T, d_out) / d_out.shape[0]
        d_biases = np.mean(d_out, axis=0, keepdims=True)
        d_input = np.dot(d_out, self.weights.T)
        
        self.weights -= learning_rate * d_weights
        self.biases -= learning_rate * d_biases
        
        return d_input


class ReLU:
    """ReLU activation"""
    
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask
    
    def backward(self, d_out):
        return d_out * self.mask


class Softmax:
    """Softmax activation for output"""
    
    def forward(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        self.output = exp_x / np.sum(exp_x, axis=1, keepdims=True)
        return self.output
    
    def backward(self, y_true):
        """Combined with cross-entropy loss"""
        return self.output - y_true

print("✅ Helper layers created!")

## 📊 Part 5: Load Dataset & Train

In [ ]:
# Load digits dataset (8x8 images, 10 classes)
digits = load_digits()
X, y = digits.data, digits.target

# Reshape to (samples, 8, 8, 1) for CNN
X = X.reshape(-1, 8, 8, 1)

# Normalize
X = X / 16.0

# One-hot encode labels
num_classes = 10
y_onehot = np.zeros((len(y), num_classes))
y_onehot[np.arange(len(y)), y] = 1

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42
)

print(f"Dataset: Digits (8x8 images)")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Image shape: {X_train.shape[1:]}")
print(f"Classes: {num_classes}")

# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    row, col = i // 5, i % 5
    idx = np.where(np.argmax(y_train, axis=1) == i)[0][0]
    axes[row, col].imshow(X_train[idx, :, :, 0], cmap='gray')
    axes[row, col].set_title(f'Label: {i}', fontsize=12)
    axes[row, col].axis('off')
plt.suptitle('Sample Digits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🚀 Training the CNN

In [ ]:
# Build CNN: Conv -> ReLU -> MaxPool -> Flatten -> Dense -> Softmax

# Layer 1: Conv (1 -> 8 filters, 3x3 kernel)
conv1 = Conv2D(num_filters=8, kernel_size=3, input_channels=1, stride=1, padding=1)
relu1 = ReLU()
pool1 = MaxPool2D(pool_size=2, stride=2)

# After conv: (8, 8, 8), after pool: (4, 4, 8)
# Flatten: 4 * 4 * 8 = 128
flatten = Flatten()

# Dense layers
dense1 = Dense(128, 64)
relu2 = ReLU()
dense2 = Dense(64, 10)
softmax = Softmax()

print("CNN Architecture:")
print("Input: (8, 8, 1)")
print("Conv1: (8, 8, 8) -> ReLU -> Pool: (4, 4, 8)")
print("Flatten: 128")
print("Dense1: 128 -> 64 -> ReLU")
print("Dense2: 64 -> 10 -> Softmax")
print(f"\nTotal conv parameters: {conv1.kernels.size + conv1.biases.size}")
print(f"Total dense parameters: {dense1.weights.size + dense1.biases.size + dense2.weights.size + dense2.biases.size}")

# Training function
def train_cnn(X, y, epochs=50, batch_size=32, lr=0.01):
    losses = []
    accuracies = []
    
    for epoch in range(epochs):
        # Shuffle
        indices = np.random.permutation(X.shape[0])
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        epoch_loss = 0
        num_batches = 0
        
        for i in range(0, X.shape[0], batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]
            
            # Forward pass
            out = conv1.forward(X_batch)
            out = relu1.forward(out)
            out = pool1.forward(out)
            out = flatten.forward(out)
            out = dense1.forward(out)
            out = relu2.forward(out)
            out = dense2.forward(out)
            out = softmax.forward(out)
            
            # Loss (cross-entropy)
            loss = -np.mean(np.sum(y_batch * np.log(out + 1e-15), axis=1))
            epoch_loss += loss
            num_batches += 1
            
            # Backward pass
            d_out = softmax.backward(y_batch)
            d_out = dense2.backward(d_out, lr)
            d_out = relu2.backward(d_out)
            d_out = dense1.backward(d_out, lr)
            d_out = flatten.backward(d_out)
            d_out = pool1.backward(d_out)
            d_out = relu1.backward(d_out)
            d_out = conv1.backward(d_out, lr)
        
        avg_loss = epoch_loss / num_batches
        losses.append(avg_loss)
        
        # Accuracy
        out = conv1.forward(X)
        out = relu1.forward(out)
        out = pool1.forward(out)
        out = flatten.forward(out)
        out = dense1.forward(out)
        out = relu2.forward(out)
        out = dense2.forward(out)
        out = softmax.forward(out)
        
        preds = np.argmax(out, axis=1)
        true = np.argmax(y, axis=1)
        acc = np.mean(preds == true)
        accuracies.append(acc)
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | Loss: {avg_loss:.4f} | Accuracy: {acc:.4f}")
    
    return losses, accuracies

print("\n🚀 Starting training...\n")
losses, accuracies = train_cnn(X_train, y_train, epochs=100, batch_size=32, lr=0.01)
print("\n✅ Training complete!")

## 📈 Results & Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses, color='red', linewidth=2)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(accuracies, color='green', linewidth=2)
axes[1].set_title('Training Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Test accuracy
out = conv1.forward(X_test)
out = relu1.forward(out)
out = pool1.forward(out)
out = flatten.forward(out)
out = dense1.forward(out)
out = relu2.forward(out)
out = dense2.forward(out)
out = softmax.forward(out)

test_preds = np.argmax(out, axis=1)
test_true = np.argmax(y_test, axis=1)
test_acc = np.mean(test_preds == test_true)

print(f"\n📊 Final Results:")
print(f"   Training Accuracy: {accuracies[-1]:.4f}")
print(f"   Test Accuracy: {test_acc:.4f}")

## 🔍 Visualizing Learned Filters

In [ ]:
# Visualize the learned convolution filters
filters = conv1.kernels[:, :, 0, :]  # Shape: (3, 3, num_filters)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(8):
    row, col = i // 4, i % 4
    axes[row, col].imshow(filters[:, :, i], cmap='RdBu_r', interpolation='nearest')
    axes[row, col].set_title(f'Filter {i+1}', fontsize=10)
    axes[row, col].axis('off')

plt.suptitle('Learned Convolution Filters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Visualize feature maps for a test image
sample_img = X_test[0:1]  # First test image

# Get feature maps
feat_maps = conv1.forward(sample_img)
feat_maps = relu1.forward(feat_maps)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(8):
    row, col = i // 4, i % 4
    axes[row, col].imshow(feat_maps[0, :, :, i], cmap='viridis')
    axes[row, col].set_title(f'Feature Map {i+1}', fontsize=10)
    axes[row, col].axis('off')

plt.suptitle('Feature Maps for Test Image', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🎯 Exercises

1. **Add more conv layers**: Try Conv -> Pool -> Conv -> Pool -> Dense
2. **Different kernel sizes**: Test 5x5 kernels instead of 3x3
3. **Different pooling**: Compare MaxPool vs AveragePool
4. **Batch normalization**: Add batch norm after conv layers
5. **Dropout**: Add dropout for regularization
6. **Different dataset**: Try MNIST or CIFAR-10 with PyTorch

## 📚 Key Takeaways

| Concept | What We Learned |
|---------|----------------|
| **Convolution** | Sliding kernel over image to extract features |
| **Padding** | Preserves spatial dimensions |
| **Stride** | Controls downsampling |
| **Max Pooling** | Reduces size, keeps important features |
| **Feature Maps** | Different filters detect different patterns |
| **Parameter Sharing** | Same kernel used across entire image |

**Next:** Try `03-rnn-lstm.ipynb` for sequential data! 🚀